# Bootstrap Tables
This workbook downloads the base tables (locations, units, attributes, etc.) from the TEEHR-Cloud data warehouse and loads into the connected warehouse.  You must clone the repo into TEEHR-HUB and have write permissions to the data warehouse to run this.

Note that the `secrets/secrets.remote.yaml` file must exist with a valid `jupyter-user-secrets/data/api-key` TEEHR-Hub API key.

In [ ]:
import os
from pathlib import Path
import yaml

import teehr
from teehr import RemoteReadWriteEvaluation
from teehr.evaluation.spark_session_utils import create_spark_session
from teehr.utilities.apply_migrations import evolve_catalog_schema

#### Verify that the API key secret is set

In [ ]:
SECRETS = yaml.safe_load(
    Path().cwd().parents[1].joinpath("secrets", "secrets.remote.yaml").read_text()
)["secrets"]
API_KEY = SECRETS["jupyter-user-secrets"]["data"]["api-key"]
if not API_KEY:
    raise ValueError("Jupyter-user api-key secret must be populated with a valid TEEHR-Hub API key.")

#### Start the spark session

In [ ]:
spark = create_spark_session(
    aws_profile="default",
    # aws_region="us-east-1",
    update_configs={
        "spark.sql.catalog.iceberg.client.region": "us-east-1",
    }
)

#### Initialize a TEEHR Evaluation with read/write privileges

In [ ]:
ev = RemoteReadWriteEvaluation(spark=spark, enable_spark_proxy=True)

#### Configure downloads to use API key

In [ ]:
ev.download.configure(api_key=API_KEY)

#### Now you can pull in a subset of data from the TEEHR-Cloud warehouse via the API
Note: If you encounter a time out error, try reducing the page_size arg

Download domain variables, location data, and attribute data

In [ ]:
# Handpicked sites that seemed interesting
DEV_LOCATION_ID_LIST = [
    # CONUS
    "usgs-02424000",
    "usgs-03068800",
    "usgs-01570500",
    "usgs-01347000",
    "usgs-05443500",
    "usgs-06770500",
    "usgs-08313000",
    "usgs-11421000",
    "usgs-14319500",
    # Alaska
    "usgs-15200280",
    "usgs-15209700",
    "usgs-15209750",
    "usgs-15214000",
    # Hawaii
    "usgs-16010000",
    "usgs-16019000",
    "usgs-16031000",
    "usgs-16060000",
    # Puerto Rico
    "usgs-50010500",
    "usgs-50011000",
    "usgs-50011085",
    "usgs-50011128"
]

In [ ]:
print("Loading configurations.")
ev.download.configurations(load=True)

print("Loading units.")
ev.download.units(load=True)

print("Loading variables.")
ev.download.variables(load=True)

print("Loading locations.")
ev.download.locations(
    ids=DEV_LOCATION_ID_LIST,
    load=True
)
print("Loading location croswalks.")
ev.download.location_crosswalks(
    primary_location_id=DEV_LOCATION_ID_LIST,
    load=True
)
print("Loading location attributes.")
ev.download.attributes(load=True)
ev.download.location_attributes(
    location_id=DEV_LOCATION_ID_LIST,
    load=True
)
print("Loading domain variables, locations, crosswalks, and attributes complete!")

In [ ]:
spark.stop()